# VAE for EHR Code Sequences

This is one notebook in a series of three about VAEs:

1. [Synthetic image VAE](../cxr/vae_cxr_synthetic.ipynb)
2. [ChestXray14 image VAE](../cxr/vae_cxr_chestxray14.ipynb)
3. [EHR code-sequence VAE (this notebook)](vae_ehr_sequences.ipynb)

Its inputs are categorical EHR event sequences (for example, diagnosis codes), not numeric measurements such as vital signs. This example uses small synthetic sequences by default, so it runs without MIMIC access.


## What is a VAE, and why use one for EHR codes?

A **variational autoencoder (VAE)** learns a probability distribution over a compact latent representation of each input. It samples a latent vector and decodes it to reconstruct the input. Training balances reconstruction loss against a KL-divergence penalty that keeps latent distributions near a simple prior, encouraging a smooth latent space that can also be sampled to generate new examples.

For categorical EHR sequences, the encoder summarizes the ordered code tokens and an autoregressive decoder predicts them back one at a time. Potential research uses include learning patient representations, unsupervised pretraining, and studying synthetic code sequences. This example handles categorical sequences; it does not model numeric measurements, elapsed time, or full visit structure. Generated codes are not validated patient records and should not be treated as clinical recommendations or realistic synthetic data without further evaluation.

In PyHealth, the `SequenceProcessor` turns codes into vocabulary indices, while the VAE can train through the usual dataset and `Trainer` interfaces. This makes the model a reusable research baseline; the small synthetic example checks the workflow, not clinical usefulness.

## Build a synthetic sequence dataset

`SequenceProcessor` maps clinical-code strings to token IDs and pads batches. The mortality label is present only to mirror a common EHR task; the VAE learns to reconstruct its input sequence.


In [1]:
import torch
from pyhealth.datasets import create_sample_dataset, get_dataloader
from pyhealth.models import VAE
from pyhealth.trainer import Trainer

codes = ["I10", "E11", "J18", "N17", "I50"]
samples = [
    {"patient_id": f"p{i}", "conditions": codes[i % 5 : i % 5 + 1] + codes[: i % 3], "mortality": i % 2}
    for i in range(20)
]
dataset = create_sample_dataset(
    samples=samples,
    input_schema={"conditions": "sequence"},
    output_schema={"mortality": "binary"},
    dataset_name="vae-ehr-sequences-demo",
)
loader = get_dataloader(dataset, batch_size=4, shuffle=True, seed=7)
batch = next(iter(loader))
print("Padded token IDs:", batch["conditions"].shape)
print("Vocabulary:", dataset.input_processors["conditions"].code_vocab)

W1030 14:56:04.546000 58740 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Label mortality vocab: {0: 0, 1: 1}


Padded token IDs: torch.Size([4, 3])
Vocabulary: {'<pad>': 0, '<unk>': 1, 'I10': 2, 'E11': 3, 'J18': 4, 'N17': 5, 'I50': 6}


## Train and check reconstruction

The encoder summarizes each variable-length sequence. The decoder predicts the next token at each position; padding positions are excluded from the reconstruction loss. Token loss is summed within each sequence and then averaged across sequences, matching the per-sequence average used for KL loss.


In [2]:
model = VAE(dataset, feature_key="conditions", latent_dim=8, hidden_dim=8)
trainer = Trainer(
    model=model,
    device="cuda" if torch.cuda.is_available() else "cpu",
    metrics=[],
)
trainer.train(
    train_dataloader=loader,
    epochs=50,
    optimizer_params={"lr": 1e-3},
)
with torch.no_grad():
    result = model(**batch)
print("Token-logit shape:", result["reconstruction_logits"].shape)
print("VAE objective (reconstruction + KL):", float(result["loss"]))

VAE(
  (embedding_model): EmbeddingModel(embedding_layers=ModuleDict(
    (conditions): Embedding(7, 8, padding_idx=0)
  ))
  (sequence_encoder): GRU(8, 8, batch_first=True)
  (sequence_mu): Linear(in_features=8, out_features=8, bias=True)
  (sequence_log_var): Linear(in_features=8, out_features=8, bias=True)
  (latent_to_hidden): Linear(in_features=8, out_features=8, bias=True)
  (sequence_decoder): GRU(8, 8, batch_first=True)
  (sequence_output): Linear(in_features=8, out_features=7, bias=True)
)


Metrics: []


Device: cpu


Training:


Batch size: 4


...

Epoch 49 / 50:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 49 / 50: 100%|██████████| 5/5 [00:00<00:00, 1364.36it/s]

--- Train epoch-49, step-250 ---


loss: 2.7061


Token-logit shape: torch.Size([4, 3, 7])
VAE objective (reconstruction + KL): 3.1402482986450195


## Generate fixed-length code sequences

Generation is autoregressive and returns token IDs. The notebook maps those IDs back to the fitted vocabulary for inspection. Generated codes are a software demonstration, not validated synthetic clinical data.


In [3]:
model.eval()
with torch.no_grad():
    generated = model.generate(num_samples=3, sequence_length=5)

vocab = dataset.input_processors["conditions"].code_vocab
token_by_id = {index: token for token, index in vocab.items()}
decoded = [[token_by_id[int(token)] for token in row] for row in generated.cpu()]
decoded

[['E11', 'E11', 'I10', 'I10', 'I10'],
 ['N17', 'E11', 'I10', 'I10', 'I10'],
 ['E11', 'I10', 'E11', 'N17', 'I10']]

## Optional: use MIMIC-IV demo data

Set `PYHEALTH_MIMIC4_ROOT` to a local MIMIC-IV or MIMIC-IV demo directory to build the same sequence model from the built-in mortality task. Credentialed data stays on your machine.


In [4]:
import os
from pathlib import Path

mimic_root = os.environ.get("PYHEALTH_MIMIC4_ROOT")
if mimic_root and Path(mimic_root).exists():
    from pyhealth.datasets import MIMIC4Dataset, get_dataloader
    from pyhealth.tasks import MortalityPredictionMIMIC4

    ehr = MIMIC4Dataset(
        ehr_root=mimic_root,
        ehr_tables=["diagnoses_icd", "procedures_icd", "prescriptions"],
        dev=True,
    )
    ehr_samples = ehr.set_task(MortalityPredictionMIMIC4())
    ehr_model = VAE(
        ehr_samples, feature_key="conditions", latent_dim=32, hidden_dim=32
    )
    ehr_loader = get_dataloader(ehr_samples, batch_size=16, shuffle=True)
    ehr_trainer = Trainer(model=ehr_model, metrics=[])
    ehr_trainer.train(train_dataloader=ehr_loader, epochs=1)
else:
    print("Set PYHEALTH_MIMIC4_ROOT to run the optional real-data example.")


Set PYHEALTH_MIMIC4_ROOT to run the optional real-data example.
